# Logistic regression feature search — Version 4

In [1]:
# %pip install optuna "scikit-learn>=1.4" scipy joblib
import numpy as np
import pandas as pd
import optuna
import joblib
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from preprocessing import import_data, group_diagnoses, group_medications, engineer, features

N_TRIALS = 5 #3_500 additional trials per execution
STORAGE = "sqlite:///logistic_regression_Version_4.db"
STUDY_NAME = "logistic_f2_individual_features_v4"

c:\Users\daoke\Documents\GitHub\fall-2026-hospital-readmission-for-diabetes\.venv314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Candidates

In [2]:
Xt, yt, Xv, yv = import_data()
base_numeric = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
                "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
visits = ["number_outpatient", "number_emergency", "number_inpatient"]
diags = ["diag_1", "diag_2", "diag_3"]
drugs = Xt.loc[:, "metformin":"metformin-pioglitazone"].columns.tolist()
classes = {
    "meglitinide": ["repaglinide", "nateglinide"],
    "sulfonylurea_gen1": ["chlorpropamide", "acetohexamide", "tolbutamide", "tolazamide"],
    "sulfonylurea_gen2": ["glimepiride", "glipizide", "glyburide"],
    "thiazolidinedione": ["pioglitazone", "rosiglitazone", "troglitazone"],
    "alpha_glucosidase": ["acarbose", "miglitol"],
    "metformin_combinations": ["glyburide-metformin", "glipizide-metformin",
                                "metformin-rosiglitazone", "metformin-pioglitazone"],
}

frames = []
for raw in [Xt, Xv]:
    X = raw.drop(columns=["encounter_id", "readmitted", "target", "patient_nbr"], errors="ignore").copy()
    grouped_diag = group_diagnoses(X)
    grouped_med = group_medications(X)
    for col in diags:
        X[col + "_group"] = grouped_diag[col]
    X = pd.concat([X, grouped_med.loc[:, grouped_med.columns.difference(X.columns)]], axis=1)
    X = engineer(X)
    extra = {}
    total = X[visits].sum(axis=1, min_count=3)
    extra["eng_total_visits"] = total
    extra["eng_acute_visits"] = X["number_inpatient"] + X["number_emergency"]
    extra["eng_any_prior_visit"] = total.gt(0).astype(int)
    extra["eng_utilization_types"] = X[visits].gt(0).sum(axis=1)
    extra["eng_acute_visit_share"] = extra["eng_acute_visits"] / (1 + total)
    extra["eng_emergency_inpatient_ratio"] = X["number_emergency"] / (1 + X["number_inpatient"])
    for col in visits:
        z = X[col]
        extra[col + "_bin"] = pd.cut(z, [-1, 0, 1, 2, 4, np.inf],
                                     labels=["0", "1", "2", "3-4", "5+"]).astype(object)
        extra[col + "_share"] = z / (1 + total)
        extra[col + "_2plus"] = z.ge(2).astype(int)
        extra[col + "_5plus"] = z.ge(5).astype(int)
    for col in base_numeric:
        z = X[col].clip(lower=0)
        extra[col + "_log1p"] = np.log1p(z)
        extra[col + "_sqrt"] = np.sqrt(z)
        extra[col + "_squared"] = z ** 2
    for i, a in enumerate(base_numeric):
        for b in base_numeric[i + 1:]:
            extra[a + "_x_" + b] = extra[a + "_log1p"] * extra[b + "_log1p"]
    days = X["time_in_hospital"].clip(lower=1)
    for col in ["num_procedures", "number_diagnoses", "num_medications", "num_lab_procedures"]:
        extra[col + "_per_day"] = X[col] / days
    extra["eng_labs_per_medication"] = X["num_lab_procedures"] / (1 + X["num_medications"])
    for status in ["No", "Steady", "Up", "Down"]:
        extra["eng_drugs_" + status] = X[drugs].eq(status).sum(axis=1)
    extra["eng_drugs_active"] = X[drugs].isin(["Steady", "Up", "Down"]).sum(axis=1)
    extra["eng_drugs_changed"] = X[drugs].isin(["Up", "Down"]).sum(axis=1)
    extra["eng_drugs_missing"] = X[drugs].isna().sum(axis=1)
    for drug in drugs:
        extra[drug + "_active"] = X[drug].isin(["Steady", "Up", "Down"]).astype(int)
        extra[drug + "_changed"] = X[drug].isin(["Up", "Down"]).astype(int)
    for name, members in classes.items():
        for status in ["Steady", "Up", "Down"]:
            extra[name + "_count_" + status] = X[members].eq(status).sum(axis=1)
        extra[name + "_count_active"] = X[members].isin(["Steady", "Up", "Down"]).sum(axis=1)
    diag_group = X[[c + "_group" for c in diags]]
    extra["eng_distinct_diagnosis_groups"] = diag_group.nunique(axis=1)
    for group in ["Diabetes", "Circulatory", "Respiratory", "Digestive",
                  "Genitourinary", "Musculoskeletal", "Neoplasms", "Injury"]:
        extra["eng_diagnosis_count_" + group] = diag_group.eq(group).sum(axis=1)
    for col in diags:
        extra[col + "_prefix"] = X[col].astype("string").str.extract(r"^([EV]?\d{1,3})", expand=False)
    age_mid = pd.to_numeric(X["age"].astype("string").str.extract(r"\[(\d+)-", expand=False), errors="coerce") + 5
    extra["eng_age_midpoint"] = age_mid
    extra["eng_age_squared"] = age_mid ** 2
    for cutoff in [50, 65, 75, 85]:
        extra[f"eng_age_{cutoff}plus"] = age_mid.ge(cutoff).astype(int)
    for col in ["a1cresult", "max_glu_serum"]:
        extra[col + "_tested"] = X[col].ne("None").astype(int)
    for a, b in [("diag_1_group", "a1cresult"), ("diag_1_group", "insulin"),
                 ("diag_1_group", "discharge_disposition_id"), ("age", "discharge_disposition_id"),
                 ("insulin", "change"), ("admission_type_id", "admission_source_id"),
                 ("diag_1_group", "diag_2_group"), ("diabetesmed", "a1cresult")]:
        extra[a + "_with_" + b] = X[a].fillna("Missing").astype(str) + "|" + X[b].fillna("Missing").astype(str)
    X = pd.concat([X, pd.DataFrame({k: v for k, v in extra.items() if k not in X.columns},
                                   index=X.index)], axis=1)
    for col in diags + ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
        X[col] = X[col].astype(str).where(X[col].notna(), np.nan)
    for col in X.select_dtypes(exclude="number"):
        X[col] = X[col].astype(str).where(X[col].notna(), np.nan)
    frames.append(X)
Xt, Xv = frames
all_features = Xt.columns.tolist()
baseline_features = ["diag_1_group" if col == "diag_1" else col for col in features()]
print(f"{len(all_features)} individually switchable candidate features")

Importing train/validation data...
Finished the import.
245 individually switchable candidate features


## Encode once and select Features after

In [3]:
transformers = []
for i, col in enumerate(all_features):
    if pd.api.types.is_numeric_dtype(Xt[col]):
        prep = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True,
                                                  add_indicator=True)),
                         ("scale", StandardScaler())])
    else:
        prep = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Missing",
                                                  keep_empty_features=True)),
                         ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                  min_frequency=20))])
    transformers.append((f"feature_{i}", prep, [col]))
preprocessor = ColumnTransformer(transformers, sparse_threshold=1.0)
M_train = sparse.csc_matrix(preprocessor.fit_transform(Xt))
M_valid = sparse.csc_matrix(preprocessor.transform(Xv))
feature_columns = {}
for i, col in enumerate(all_features):
    sl = preprocessor.output_indices_[f"feature_{i}"]
    feature_columns[col] = np.arange(sl.start, sl.stop)
print("Cached encoded shapes:", M_train.shape, M_valid.shape)

Cached encoded shapes: (65129, 2293) (16283, 2293)


## Search and resume

In [4]:
def objective(trial):
    chosen = [col for col in all_features
              if trial.suggest_categorical("use_" + col, [False, True])]
    C = trial.suggest_float("C", 1e-4, 40, log=True)
    weight_type = trial.suggest_categorical("class_weight", ["none", "balanced", "custom"])
    weight = ({0: 1, 1: trial.suggest_float("positive_weight", 1, 50, log=True)}
              if weight_type == "custom" else "balanced" if weight_type == "balanced" else None)
    if not chosen:
        return 0.0
    selected = np.concatenate([feature_columns[col] for col in chosen])
    model = LogisticRegression(solver="liblinear", C=C,
                               class_weight=weight, max_iter=300, tol=1e-3)
    model.fit(M_train[:, selected].tocsr(), yt)
    p = model.predict_proba(M_valid[:, selected].tocsr())[:, 1]
    precision, recall, thresholds = precision_recall_curve(yv, p)
    f2 = 5 * precision[:-1] * recall[:-1] / (4 * precision[:-1] + recall[:-1] + 1e-15)
    k = int(f2.argmax())
    trial.set_user_attr("features", chosen)
    trial.set_user_attr("n_features", len(chosen))
    trial.set_user_attr("threshold", float(thresholds[k]))
    return float(f2[k])

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(study_name=STUDY_NAME, storage=STORAGE,
                            direction="maximize", load_if_exists=True,
                            sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=30))
if not study.trials:
    study.enqueue_trial({**{"use_" + col: True for col in all_features},
                         "C": 0.01573, "class_weight": "balanced"})
    study.enqueue_trial({**{"use_" + col: col in baseline_features for col in all_features},
                         "C": 0.2, "class_weight": "custom", "positive_weight": 22.0})
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1,
               callbacks=[lambda s, t: print(f"{t.number + 1} stored trials, best F2 {s.best_value:.5f}")
                          if (t.number + 1) % 25 == 0 else None])

## Inspect and save

In [5]:
best = study.best_trial
print(f"Best tuned validation F2: {best.value:.5f}")
print(f"Selected {best.user_attrs['n_features']} inputs")
print("Threshold:", best.user_attrs["threshold"])
print("Features:", best.user_attrs["features"])
print("C and class weight:", {k: v for k, v in best.params.items() if not k.startswith("use_")})
results = study.trials_dataframe()
results.to_csv("logistic_regression_Version_4_trials.csv", index=False)
display(results.sort_values("value", ascending=False).head(10)[
    ["number", "value", "duration", "user_attrs_n_features", "user_attrs_threshold"]])
chosen = best.user_attrs["features"]
selected = np.concatenate([feature_columns[col] for col in chosen])
weight_type = best.params["class_weight"]
weight = ({0: 1, 1: best.params["positive_weight"]} if weight_type == "custom" else
          "balanced" if weight_type == "balanced" else None)
model = LogisticRegression(solver="liblinear", C=best.params["C"],
                           class_weight=weight, max_iter=300, tol=1e-3)
model.fit(M_train[:, selected].tocsr(), yt)
joblib.dump({"preprocessor": preprocessor, "model": model, "features": chosen,
             "encoded_columns": selected, "threshold": best.user_attrs["threshold"],
             "validation_f2": best.value, "candidate_names": all_features},
            "logistic_regression_Version_4_best.joblib")

Best tuned validation F2: 0.43951
Selected 124 inputs
Threshold: 0.4483490425281894
Features: ['race', 'age', 'admission_type_id', 'payer_code', 'medical_specialty', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'metformin-pioglitazone', 'diabetesmed', 'alpha_glucosidase_inhibitor', 'biguanide', 'sulfonylurea_gen2', 'thiazolidinedione', 'number_outpatient_log1p', 'number_emergency_log1p', 'number_inpatient_log1p', 'eng_prior_inpatient_any', 'eng_inpatient_emergency_interaction', 'eng_labs_per_day', 'eng_acute_visits', 'eng_any_prior_visit', 'eng_acute_visit_share', 'number_emergency_bin', 'number_emergency_2plus', 'number_inpatient_2plus', 'number_inpatient_5plus', 'time_in_hospital_log1p', 'time_in_hospital_sqrt', 'time_in_hospital_squared', 'num_lab_procedures_log1p', 'num_lab_procedures_sqrt', 'num_lab_procedures_squared', 'num_procedures_log1p', 'num_proce

,number,value,duration,user_attrs_n_features,user_attrs_threshold
292,292,0.439513,0 days 00:00:27.698589,124.0,0.448349
420,420,0.439503,0 days 00:00:42.370754,126.0,0.443536
603,603,0.439415,0 days 00:01:08.048267,128.0,0.442536
1367,1367,0.439411,0 days 00:00:52.803127,127.0,0.443918
672,672,0.439366,0 days 00:01:08.802912,129.0,0.440986
1423,1423,0.439332,0 days 00:00:53.384678,119.0,0.442816
949,949,0.439277,0 days 00:01:11.197092,120.0,0.445232
758,758,0.439224,0 days 00:00:49.529300,126.0,0.443651
901,901,0.439220,0 days 00:01:13.353195,127.0,0.441405
1392,1392,0.439196,0 days 00:00:33.728902,121.0,0.444476


['logistic_regression_Version_4_best.joblib']

# Best Model after Tuning

In [6]:
best.user_attrs['features']

['race',
 'age',
 'admission_type_id',
 'payer_code',
 'medical_specialty',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'metformin-pioglitazone',
 'diabetesmed',
 'alpha_glucosidase_inhibitor',
 'biguanide',
 'sulfonylurea_gen2',
 'thiazolidinedione',
 'number_outpatient_log1p',
 'number_emergency_log1p',
 'number_inpatient_log1p',
 'eng_prior_inpatient_any',
 'eng_inpatient_emergency_interaction',
 'eng_labs_per_day',
 'eng_acute_visits',
 'eng_any_prior_visit',
 'eng_acute_visit_share',
 'number_emergency_bin',
 'number_emergency_2plus',
 'number_inpatient_2plus',
 'number_inpatient_5plus',
 'time_in_hospital_log1p',
 'time_in_hospital_sqrt',
 'time_in_hospital_squared',
 'num_lab_procedures_log1p',
 'num_lab_procedures_sqrt',
 'num_lab_procedures_squared',
 'num_procedures_log1p',
 'num_procedures_sqrt',
 'num_medications_log1p',
 'num_medica

In [7]:
best.user_attrs['n_features']

124

In [8]:
print("C and class weight:", {k: v for k, v in best.params.items() if not k.startswith("use_")})

C and class weight: {'C': 1.5036376551308464, 'class_weight': 'balanced'}
